# Dispositivos nuevos o no inventariados

## Objetivo

Identificar dispositivos nuevos observados por DHCP y revisar su comportamiento DNS inicial.

## Entradas esperadas

- Ventana de linea base.
- Ventana reciente de deteccion.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. Nuevas direcciones MAC o hostnames.
2. Primeras IPs observadas.
3. Actividad DNS inicial.
4. Priorizacion de riesgo.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_new_devices = """
let BaselineWindow = 30d;
let DetectionWindow = 1d;
let baseline =
    fn_Normalize_Windows_DHCP(BaselineWindow + DetectionWindow)
    | where TimeGenerated between (ago(BaselineWindow + DetectionWindow) .. ago(DetectionWindow))
    | summarize by ClientMac, HostName;
fn_Normalize_Windows_DHCP(DetectionWindow)
| where isnotempty(ClientMac) or isnotempty(HostName)
| join kind=leftanti baseline on ClientMac, HostName
| summarize FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated), Ips=make_set(ClientIp, 20), DhcpServers=make_set(DeviceName, 10) by ClientMac, HostName
| order by FirstSeen desc
"""
# new_devices_df = qry_prov.exec_query(query_new_devices)
print(query_new_devices)


In [ ]:
query_new_devices_dns = """
let BaselineWindow = 30d;
let DetectionWindow = 1d;
let baseline =
    fn_Normalize_Windows_DHCP(BaselineWindow + DetectionWindow)
    | where TimeGenerated between (ago(BaselineWindow + DetectionWindow) .. ago(DetectionWindow))
    | summarize by ClientMac, HostName;
let new_hosts =
    fn_Normalize_Windows_DHCP(DetectionWindow)
    | where isnotempty(ClientIp)
    | join kind=leftanti baseline on ClientMac, HostName
    | summarize by ClientMac, HostName;
fn_Correlate_DHCP_DNS(DetectionWindow)
| where ClientMac in (new_hosts | project ClientMac) or HostName in (new_hosts | project HostName)
| summarize TotalDns=count(), NxdomainCount=countif(toupper(ResponseCode) has_any ("NXDOMAIN", "NAME_ERROR", "3") or RawMessage has_any ("NXDOMAIN", "Name Error")), DistinctDomains=dcount(QueryRootDomain), MaxQueryLength=max(QueryLength), SampleQueries=make_set(QueryName, 20) by ClientIp, HostName, ClientMac
| order by NxdomainCount desc, MaxQueryLength desc
"""
# new_devices_dns_df = qry_prov.exec_query(query_new_devices_dns)
print(query_new_devices_dns)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
